In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

import sys
sys.path.append('../code/graphParser')
sys.path.append('../code/ragas_custom')

from rateLimit import handle_rate_limits
from retrieve.sparse import BM25
from retrieve.config import generate_retriever_configs

from ragas.testset.graph import KnowledgeGraph

from langchain_core.documents import Document
from langchain_core.prompts import load_prompt
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy, context_precision, faithfulness
from ragas.llms.base import llm_factory
from ragas.embeddings.base import embedding_factory

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import cohere

from typing import List

from dotenv import load_dotenv
load_dotenv()

c:\Users\owner\anaconda3\envs\SportAgent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
path = '../data/rag/test_dataset.csv'

origin_dataset = pd.read_csv(path)
origin_dataset['reference_contexts'] = origin_dataset['reference_contexts'].apply(lambda x : eval(x))
origin_dataset['reference_contexts_section'] = origin_dataset['reference_contexts_section'].apply(lambda x : eval(x))

In [3]:
origin_dataset.head(2)

,user_input,reference_contexts,reference,synthesizer_name,reference_contexts_section
0,"웨이트리프팅에서 올바른 무릎 각도를 유지하는 것이 왜 중요한지, 그리고 그것이 스내...",[다만 너무 많이 굽히면 부상을 당할 수 있으므로 연습 시 적정한 각\n도를 취하는...,웨이트리프팅에서 올바른 무릎 각도를 유지하는 것은 부상을 예방하는 데 매우 중요합니...,body part,"[II. 역도의 스포츠 과학적 원리, Ⅲ. 역도경기 기술의 구조와 훈련법]"
1,어떻게 팔과 다리의 근육을 발달시켜서 역도 기술을 향상시킬 수 있나요?,[이미지에는 역도 선수가 바벨을 들어 올리는 일련의 동작이 순서대로 나타나 있다. ...,팔과 다리의 근육을 발달시켜 역도 기술을 향상시키기 위해서는 각 체력요인에 적합한 ...,body part,"[Ⅲ. 역도경기 기술의 구조와 훈련법, II. 역도의 스포츠 과학적 원리]"


In [4]:
kg = KnowledgeGraph.load('../data/rag/kg.json')

kiwi_pos = BM25(k=20, type='kiwi_pos')
texts = [node.properties['page_content'] for node in kg.nodes]
kiwi_pos.from_texts(texts)

documents = [Document(page_content=node.properties['page_content'],
                      metadata=node.properties['document_metadata'])
                       for node in kg.nodes]

embeddings = OpenAIEmbeddings()
db = FAISS.from_documents(documents, embeddings)

# 1. Retrieve

In [5]:
def precompute(dense, sparse, alpha, threshold, k=20):
    results = []
    normalized_alpha = alpha / 100.0

    over_threshold = []
    for document_list in dense:
        tmp = []
        for doc, score in document_list:
            if score >= threshold:
                tmp.append(doc.page_content)

        over_threshold.append(tmp)

    for dense_list, sparse_list in zip(over_threshold, sparse):   
        doc_scores = {}
        for i, doc in enumerate(dense_list):
            rank_score = 1.0 / (i + 1)
            doc_scores[doc] = normalized_alpha * rank_score
        for i, doc in enumerate(sparse_list):
            rank_score = 1.0 / (i + 1)
            if doc in doc_scores:
                doc_scores[doc] += (1 - normalized_alpha) * rank_score
            else:
                doc_scores[doc] = (1 - normalized_alpha) * rank_score

        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:k]
        sorted_docs = [doc for doc, score in sorted_docs]
        results.append(sorted_docs)

    return results

In [10]:
origin_dataset['precompute_dense'] = origin_dataset['user_input'].apply(lambda x : db.similarity_search_with_score(x, k=int(20)))
origin_dataset['precompute_sparse_bm25_kiwi_pos'] = origin_dataset['user_input'].apply(lambda x : kiwi_pos.search(x))

origin_dataset['retrieved_contexts'] = precompute(origin_dataset['precompute_dense'], origin_dataset['precompute_sparse_bm25_kiwi_pos'], 40, 0.1)

In [ ]:
# origin_dataset.to_csv('../data/rag/precompute_testset.csv', index=False)

# 2. Rerank

In [ ]:
co = cohere.ClientV2()

# origin_dataset['cohere'] = origin_dataset.progress_apply(lambda x : co.rerank(model='rerank-v3.5', query=x['user_input'], documents=x['retrieved_contexts']), axis = 1)
origin_dataset['cohere_contexts'] = origin_dataset.apply(lambda x : [result.index for result in x['cohere'].results], axis=1)
origin_dataset['cohere_contexts'] = origin_dataset.apply(lambda x: [x['retrieved_contexts'][i] for i in x['cohere_contexts']], axis=1)
origin_dataset['rerank'] = origin_dataset.apply(lambda x : x['cohere_contexts'][:9], axis=1)

# origin_dataset.to_csv('../data/rag/precompute_testset.csv', index=False)

100%|██████████| 22/22 [00:07<00:00,  2.95it/s]


# 3. Generate

In [ ]:
origin_dataset = pd.read_csv('../data/rag/precompute_testset.csv')

for column in ['reference_contexts', 'retrieved_contexts', 'rerank']:
    origin_dataset[column] = origin_dataset[column].apply(lambda x : eval(x))

In [6]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.4)

ragas_llm = llm_factory('gpt-4o-mini')
ragas_embedding = embedding_factory()

@handle_rate_limits
def generate_chain(data_batches, llm, current_api_key=None, max_concurrency=5) -> List:
    prompt = load_prompt('../prompt/generate/generate.yaml')
    chain = prompt | llm 
    results = chain.batch(data_batches, config={"max_concurrency": max_concurrency})
    return [result.content for result in results]

In [ ]:
max_concurrency = 5
input_querys = origin_dataset['user_input'].to_list()
documents = origin_dataset['rerank']

results = []

for i in tqdm(range(len(input_querys) // max_concurrency)):
    input_query_batches = input_querys[i * max_concurrency : (i + 1) * max_concurrency]
    documents_batches = documents[i * max_concurrency : (i + 1) * max_concurrency]

    data_batches = [{'input_query': input_query, 'contexts': documents} for input_query, documents in zip(input_query_batches, documents_batches)]

    results.extend(generate_chain(data_batches, llm, max_concurrency=max_concurrency))

if len(input_querys) % max_concurrency != 0:
    input_query_batches = input_querys[len(input_querys) // max_concurrency * max_concurrency:]
    documents_batches = documents[len(documents) // max_concurrency * max_concurrency:]
    
    data_batches = [{'input_query': input_query, 'contexts': documents} for input_query, documents in zip(input_query_batches, documents_batches)]
    
    results.extend(generate_chain(data_batches, llm, max_concurrency=max_concurrency))

origin_dataset[f'generate'] = results

# origin_dataset.to_csv('../data/rag/precompute_testset.csv', index=False)

100%|██████████| 4/4 [00:17<00:00,  4.30s/it]


# 4. Evaluate

In [2]:
origin_dataset = pd.read_csv('../data/rag/precompute_testset.csv')

for column in ['reference_contexts', 'retrieved_contexts', 'rerank']:
    origin_dataset[column] = origin_dataset[column].apply(lambda x : eval(x))

In [3]:
noAnswer = origin_dataset.iloc[[11, -2, -4]]
normal = origin_dataset.loc[~origin_dataset.index.isin(noAnswer.index)]

## 4-1. 답변 불가 항목
* Test Dataset의 22개 항목 중 3개의 항목이 답변 불가( 'The answer is not available in the provided context.' )
* 세 항목 모두 ragas에서 제공하는 기본 합성 데이터 생성 유형인 multi_hop_specific_query, multi_hop_abstract_query 유형
* 세 항목은 rerank 절차 후 context 내에는 reference_contexts의 context가 포함되었음에도 불구하고 답변 생성 불가한 상황

In [5]:
noAnswer

,user_input,reference_contexts,reference,synthesizer_name,reference_contexts_section,precompute_dense,precompute_sparse_bm25_kiwi_pos,retrieved_contexts,cohere,cohere_contexts,rerank,generate
11,바벨 잡는 방법이 파워 스내치에 어떻게 적용되나요?,"[바벨을 잡는 방법에는 크게 오버그립(over grip), 언더그립(under gr...","바벨 잡는 방법은 파워 스내치에 중요한 역할을 합니다. 파워 스내치를 수행할 때, ...",multi_hop_abstract_query_synthesizer,"['Ⅲ. 역도경기 기술의 구조와 훈련법', 'Ⅲ. 역도경기 기술의 구조와 훈련법']",[(Document(id='224a1ffd-a46a-4b67-942e-2722100...,"['이미지는 파워 스내치의 연속 동작을 보여주고 있으며, 네 단계로 구성되어 있다....","[이미지는 파워 스내치의 연속 동작을 보여주고 있으며, 네 단계로 구성되어 있다. ...",id='4e98fd06-a351-45e5-b7d0-1c628838fd14' resu...,"['바벨을 잡는 방법에는 크게 오버그립(over grip), 언더그립(under g...","[바벨을 잡는 방법에는 크게 오버그립(over grip), 언더그립(under gr...",The answer is not available in the provided co...
20,"개별 특성에 대한 이해가 힘 훈련, 특히 저크 기술(jerk technique)에서...",[Jerk의 기술동작은 클린동작이 끝나고 시작자세에서 바벨을 가슴위에 올려놓고\n구...,힘 훈련에서 개인의 특성을 이해하는 것은 특히 저크(jerk) 기술에서 매우 중요합...,multi_hop_specific_query_synthesizer,"['Ⅲ. 역도경기 기술의 구조와 훈련법', 'II. 역도의 스포츠 과학적 원리']",[(Document(id='5ac3f78d-510b-4189-b5ff-a89f4c7...,['이미지에는 운동 제어에 대한 설명이 포함되어 있습니다. 운동 제어는 인간이 운동...,[이미지에는 운동 제어에 대한 설명이 포함되어 있습니다. 운동 제어는 인간이 운동을...,id='972f4be0-2684-47ff-8b9d-500eea540113' resu...,['스포츠심리학은 스포츠장면에서 인간의 운동행동과 스포츠 수행에 영향을 미치\n는 ...,[스포츠심리학은 스포츠장면에서 인간의 운동행동과 스포츠 수행에 영향을 미치\n는 심...,The answer is not available in the provided co...
18,"개별 특성에 대한 이해가 힘 훈련, 특히 저크 기술(jerk technique)에서...",[Jerk의 기술동작은 클린동작이 끝나고 시작자세에서 바벨을 가슴위에 올려놓고\n구...,힘 훈련에서 개인의 특성을 이해하는 것은 특히 저크(jerk) 기술에서 매우 중요합...,multi_hop_specific_query_synthesizer,"['Ⅲ. 역도경기 기술의 구조와 훈련법', 'II. 역도의 스포츠 과학적 원리']",[(Document(id='5ac3f78d-510b-4189-b5ff-a89f4c7...,['이미지에는 운동 제어에 대한 설명이 포함되어 있습니다. 운동 제어는 인간이 운동...,[이미지에는 운동 제어에 대한 설명이 포함되어 있습니다. 운동 제어는 인간이 운동을...,id='1ab544f7-f119-40d9-a169-c458d2986abd' resu...,['스포츠심리학은 스포츠장면에서 인간의 운동행동과 스포츠 수행에 영향을 미치\n는 ...,[스포츠심리학은 스포츠장면에서 인간의 운동행동과 스포츠 수행에 영향을 미치\n는 심...,The answer is not available in the provided co...


In [6]:
noAnswer.apply(lambda x : len([retrieved for retrieved in x['rerank'] if retrieved in x['reference_contexts']]) / len(x['reference_contexts']), axis=1)

11    1.0
20    0.5
18    0.5
dtype: float64

## 4-2. 일반 항목

### 4-2-1. Sentence Transformer

In [36]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

reference = model.encode(normal['reference'].to_list())
generate = model.encode(normal['generate'].to_list())

cosine_similarities = cosine_similarity(reference, generate)

In [37]:
np.mean(cosine_similarities.diagonal())

0.87564707

### 4-2-2. RAGAS: faithfulness, relevancy, context precision

In [ ]:
result_df = pd.DataFrame()
    
max_concurrency = 5
for i in range(len(normal) // max_concurrency):
    user_input_batches = normal.iloc[i * max_concurrency : (i + 1) * max_concurrency]['user_input'].to_list()
    answer_batches = normal.iloc[i * max_concurrency : (i + 1) * max_concurrency]['generate'].to_list()
    contexts_batches = normal.iloc[i * max_concurrency : (i + 1) * max_concurrency]['rerank'].to_list()
    ground_truth_batches = normal.iloc[i * max_concurrency : (i + 1) * max_concurrency]['reference'].to_list()

    data_samples = {
        'question': user_input_batches,
        'answer': answer_batches,
        'contexts': contexts_batches,
        'ground_truth': ground_truth_batches
    }

    tmp_dataset = Dataset.from_dict(data_samples)

    score = evaluate(tmp_dataset, metrics=[answer_relevancy, context_precision, faithfulness])
    score_df = score.to_pandas()
    result_df = pd.concat([result_df, score_df], ignore_index=True)

if len(normal) % max_concurrency != 0:
    user_input_batches = normal.iloc[len(normal) // max_concurrency * max_concurrency:]['user_input'].to_list()
    answer_batches = normal.iloc[len(normal) // max_concurrency * max_concurrency:]['generate'].to_list()
    contexts_batches = normal.iloc[len(normal) // max_concurrency * max_concurrency:]['rerank'].to_list()
    ground_truth_batches = normal.iloc[len(normal) // max_concurrency * max_concurrency:]['reference'].to_list()
    
    data_samples = {
        'question': user_input_batches,
        'answer': answer_batches,
        'contexts': contexts_batches,
        'ground_truth': ground_truth_batches
    }
    
    tmp_dataset = Dataset.from_dict(data_samples)

    score = evaluate(tmp_dataset, metrics=[answer_relevancy, context_precision, faithfulness])
    score_df = score.to_pandas()
    result_df = pd.concat([result_df, score_df], ignore_index=True)

Evaluating: 100%|██████████| 12/12 [00:17<00:00,  1.46s/it]


In [9]:
print('answer_relevancy :', np.mean(result_df['answer_relevancy']))
print('context_precision :', np.mean(result_df['context_precision']))
print('faithfulness :', np.mean(result_df['faithfulness']))

answer_relevancy : 0.8444037917413064
context_precision : 0.9870640141898627
faithfulness : 0.9526315789473685
